In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
accounts=pd.read_csv(r"C:\Users\agust\OneDrive\Desktop\Startup-SAAS Growth Analytics\ravenstack_accounts.csv")
events=pd.read_csv(r"C:\Users\agust\OneDrive\Desktop\Startup-SAAS Growth Analytics\ravenstack_churn_events.csv")
usage=pd.read_csv(r"C:\Users\agust\OneDrive\Desktop\Startup-SAAS Growth Analytics\ravenstack_feature_usage.csv")
subscription=pd.read_csv(r"C:\Users\agust\OneDrive\Desktop\Startup-SAAS Growth Analytics\ravenstack_subscriptions.csv")
tickets=pd.read_csv(r"C:\Users\agust\OneDrive\Desktop\Startup-SAAS Growth Analytics\ravenstack_support_tickets.csv")

In [ ]:
for name ,df in{
    "accounts":accounts,
    "events":events,
    "usage":usage,
    'subscription':subscription,
    'tickets':tickets,
}.items():
    print(f"{name.upper()}")
    print(df.columns.tolist())

In [ ]:
accounts.info()

In [ ]:
events.info()

In [ ]:
usage.info()

In [ ]:
subscription.info()

In [ ]:
tickets.info()

In [ ]:
events['feedback_text']=events['feedback_text'].fillna('NO Feedback')

In [ ]:
subscription[['end_date','churn_flag']].head(10)

In [ ]:
subscription.groupby('churn_flag')['end_date'].count()

In [ ]:
subscription['customer_status']= \
subscription['end_date'].isna().map({
    True: 'Active',
    False: 'Churned'
})

In [ ]:
tickets['priority'].value_counts()

In [ ]:
tickets['satisfaction_score'].mean()

In [ ]:
tickets['resolution_time_hours'].mean()

In [ ]:
tickets.isnull().sum()

In [ ]:
tickets['escalation_flag'].value_counts()

In [ ]:
tickets.groupby('escalation_flag')['satisfaction_score'].mean()

In [ ]:
df = accounts.copy()

In [ ]:
df = df.merge(subscription,on="account_id",how="left")

In [ ]:
usage = usage.merge(
    subscription[['subscription_id','account_id']],
    on='subscription_id',
    how='left'
)

In [ ]:
usage.columns

In [ ]:
usage_agg = usage.groupby('account_id').agg(
    total_usage_events=('usage_id','count'),
    total_usage=('usage_count','sum'),
    avg_usage=('usage_count','mean'),
    total_duration=('usage_duration_secs','sum'),
    avg_duration=('usage_duration_secs','mean'),
    total_errors=('error_count','sum'),
    unique_features=('feature_name','nunique'),
    beta_feature_usage=('is_beta_feature','mean')
).reset_index()

In [ ]:
usage_agg.columns

In [ ]:
 usage_df= usage_agg.copy()

In [ ]:
events_df = events.copy()

In [ ]:
events_agg = events.groupby("account_id").agg(
    churned =("churn_event_id","count"),
    total_refund=("refund_amount_usd","sum"),
    upgrade_before_churn=("preceding_upgrade_flag","max"),
    downgrade_before_churn=("preceding_downgrade_flag","max"),
    reactivation_flag=("is_reactivation","max"),
    churn_reason_mode=("reason_code",lambda x: x.mode()[0] if not x.mode().empty else None)
).reset_index()
print(events_agg)

In [ ]:
events_df = events_df.merge(events_agg, on="account_id", how="left")

In [ ]:
tickets_agg = tickets.groupby("account_id").agg(
    ticket_count=("ticket_id", "count"),
    high_priority = ("priority", lambda x: (x=="high").sum()),
    medium_priority = ("priority", lambda x: (x=="medium").sum()),
    low_priority = ("priority", lambda x: (x=="low").sum())
).rename(columns={"ticket_id": "ticket_count"}).reset_index()
print(tickets_agg)

In [ ]:
tickets_df = tickets.copy()

In [ ]:
tickets_df=tickets_df.merge(tickets_agg,on="account_id", how="left")

In [ ]:
master_df=accounts.copy()

In [ ]:
master_df=master_df.merge(subscription,on="account_id",how="left")

In [ ]:
master_df=master_df.merge(usage_agg,on="account_id",how="left")

In [ ]:
print(accounts.shape)


In [ ]:
print(subscription.shape)

In [ ]:
subscription_agg = subscription.groupby("account_id").agg(
    subscription_count=("subscription_id","count"),
    total_mrr=("mrr_amount","sum"),
    total_arr=("arr_amount","sum"),
    avg_seats=("seats","mean"),
    upgrade_count=("upgrade_flag","sum"),
    downgrade_count=("downgrade_flag","sum"),
    active_subscriptions=("churn_flag", lambda x: (~x).sum()),
    churned_subscriptions=("churn_flag","sum")
).reset_index()

In [ ]:
master_df = accounts.copy()


In [ ]:
print(master_df.shape)


In [ ]:
master_df = master_df.merge(
    subscription_agg,
    on="account_id",
    how="left"
)


In [ ]:
print(master_df.shape)

In [ ]:
master_df = master_df.merge(
    usage_agg,
    on="account_id",
    how="left"
)

In [ ]:
print(master_df.shape)

In [ ]:
master_df = master_df.merge(
    tickets_agg,
    on="account_id",
    how="left"
)

In [ ]:
print(master_df.shape)

In [ ]:
master_df = master_df.merge(
    events_agg,
    on="account_id",
    how="left"
)

In [ ]:
print(master_df.shape)

In [ ]:
master_df.info()

In [ ]:
master_df.isnull().sum().sort_values(ascending=False)

In [ ]:
master_df.isnull().sum()

In [ ]:
master_df.isnull().sum()

In [ ]:
tickets_agg = tickets.groupby("account_id").agg(
    ticket_count=("ticket_id","count"),
    avg_resolution_time=("resolution_time_hours","mean"),
    avg_first_response=("first_response_time_minutes","mean"),
    avg_satisfaction=("satisfaction_score","mean"),
    escalated_tickets=("escalation_flag","sum")
).reset_index()

In [ ]:
subscription_agg = subscription.groupby("account_id").agg(
    subscription_count=("subscription_id","count"),
    total_mrr=("mrr_amount","sum"),
    total_arr=("arr_amount","sum"),
    upgrades=("upgrade_flag","sum"),
    downgrades=("downgrade_flag","sum"),
    auto_renew_count=("auto_renew_flag","sum")
).reset_index()

In [ ]:
usage_sub = usage.merge(
    subscription[["subscription_id","account_id"]],
    on="subscription_id",
    how="left"
)

In [ ]:
usage_agg = usage_sub.groupby("subscription_id").agg(
    total_usage=("usage_count","sum"),
    avg_usage=("usage_count","mean"),
    total_duration=("usage_duration_secs","sum"),
    avg_duration=("usage_duration_secs","mean"),
    total_errors=("error_count","sum"),
    unique_features=("feature_name","nunique"),
    beta_feature_usage=("is_beta_feature","sum")
).reset_index()

In [ ]:
churn_agg = events.groupby("account_id").agg(
    refund_amount=("refund_amount_usd","sum"),
    reactivations=("is_reactivation","sum"),
    preceding_upgrades=("preceding_upgrade_flag","sum"),
    preceding_downgrades=("preceding_downgrade_flag","sum")
).reset_index()

In [ ]:
master_df = accounts.copy()

master_df = master_df.merge(subscription_agg,on="account_id",how="left")

master_df = master_df.merge(usage_agg,on="account_id",how="left")

master_df = master_df.merge(tickets_agg,on="account_id",how="left")

master_df = master_df.merge(churn_agg,on="account_id",how="left")

In [ ]:
#Engagement score
master_df["engagement_score"] = (
    master_df["total_usage"] +
    master_df["unique_features"]*10 +
    master_df["total_usage_events"]
)

In [ ]:
master_df["engagement_score"].describe()

In [ ]:
#Customer Segmentation
master_df["engagement_segment"] = pd.qcut(
    master_df["engagement_score"],
    q=3,
    labels=["Low","Medium","High"]
)

In [ ]:
master_df["engagement_segment"].value_counts()

In [ ]:
#Total MRR
master_df["total_mrr"].sum()

In [ ]:
#Total ARR
master_df["total_arr"].sum()

In [ ]:
#Churn Rate
master_df["churn_flag"].mean()*100

In [ ]:
#Trial Customers
master_df["is_trial"].value_counts()

In [ ]:
#REVENUE ANALYSIS

In [ ]:
master_df.groupby("plan_tier")["total_mrr"]\
.sum()\
.sort_values(ascending=False)

In [ ]:
sns.barplot(
    data=master_df,
    x="plan_tier",
    y="total_mrr"
)

plt.show()

In [ ]:
#CHURN ANALYSIS

In [ ]:
master_df.groupby("churn_flag")[[
    "engagement_score",
    "ticket_count",
    "total_usage",
    "total_mrr"
]].mean()

In [ ]:
sns.boxplot(
    data=master_df,
    x="churn_flag",
    y="engagement_score"
)

plt.show()

In [ ]:
#CUSTOMER ENGAGEMENT ANALYSIS

In [ ]:
master_df["engagement_segment"] = pd.qcut(
    master_df["engagement_score"],
    q=3,
    labels=["Low","Medium","High"]
)

In [ ]:
master_df["engagement_segment"].value_counts()

In [ ]:
sns.histplot(
    master_df["engagement_score"],
    bins=20
)

plt.show()

In [ ]:
#CORRELAYION ANALYSIS

In [ ]:
corr_matrix = master_df.select_dtypes(include='number').corr()

corr_matrix['churn_flag'].sort_values(ascending=False)

In [ ]:
master_df.columns.tolist()

In [ ]:
master_df["churn_flag"].dtype

In [ ]:
corr_matrix.columns

In [ ]:
master_df["churn_flag"] = master_df["churn_flag"].astype(int)

In [ ]:
corr_matrix = master_df.select_dtypes(include='number').corr()

corr_matrix["churn_flag"].sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(
    master_df.select_dtypes(include='number').corr(),
    annot=True
)
plt.show()

In [ ]:
#COHORT ANALYSIS

In [ ]:
master_df["signup_month"] = pd.to_datetime(
    master_df["signup_date"]
).dt.to_period("M")

In [ ]:
cohort = master_df.groupby("signup_month").agg(
    customers=("account_id","count"),
    churn_rate=("churn_flag","mean")
).reset_index()

cohort